# Cybersecurity Incident Analyzer — Fase 1

## Análise de Incidentes de Segurança com Modelos de NLP Pré-Treinados

---

### Objetivo do Projeto

O **Cybersecurity Incident Analyzer** auxilia equipes de **SOC (Security Operations Center)** na triagem inicial de incidentes de segurança da informação.

Nesta versão da **Fase 1**, o foco fica restrito a duas tarefas práticas de NLP com modelos pré-treinados:

1. **sumarização** de relatos longos de incidentes;
2. **extração com NER** de entidades relevantes para triagem.

### O Problema

Relatos de incidentes costumam chegar em formato narrativo: tickets, e-mails, observações de analistas, alertas enriquecidos e descrições de usuários. A leitura manual desse material é:

- lenta;
- sujeita a inconsistências;
- difícil de escalar.

### Objetivo da Fase 1

Esta fase demonstra como usar modelos pré-treinados da Hugging Face para:

- resumir relatos extensos;
- identificar entidades e observáveis relevantes.

> **Fora de escopo nesta fase:** classificação zero-shot, geração de recomendações, prompt engineering, RAG, embeddings, bancos vetoriais e recuperação documental.

### Ambiente recomendado

Este notebook foi projetado para rodar no **Google Colab** com runtime de **GPU (T4 ou superior)**. Para ativar: `Ambiente de execução > Alterar o tipo de ambiente de execução > GPU`.


---
## 1. Instalação de Dependências e Imports

Instalamos as bibliotecas necessárias para carregar modelos pré-treinados, manipular o dataset e avaliar os resultados.


In [1]:
!pip install -q transformers torch pandas numpy sentencepiece accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 120.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 101.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 22.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 102.8 MB/s eta 0:00:0000:0100:01


In [2]:
import json
import textwrap
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, pipeline


device = 0 if torch.cuda.is_available() else -1
device_name = "GPU" if device == 0 else "CPU"
print(f"Dispositivo em uso: {device_name}")

if torch.cuda.is_available():
    print(f"GPU detectada: {torch.cuda.get_device_name(0)}")


Dispositivo em uso: GPU
GPU detectada: Tesla T4


In [3]:
def print_wrapped(text, width=100):
    for line in str(text).splitlines() or [""]:
        if line.strip():
            print(
                textwrap.fill(
                    line,
                    width=width,
                    break_long_words=False,
                    break_on_hyphens=False,
                )
            )
        else:
            print()


---
## 2. Dataset de Incidentes (Exemplos Fictícios)

O dataset abaixo contém **10 incidentes fictícios**. Cada exemplo inclui:

- o relato do incidente em inglês;
- comentários e explicações em português.

Nesta fase, os textos são usados para demonstrar sumarização e extração direta de entidades com NER.


In [ ]:
DATA_PATH_CANDIDATES = [
    Path("data/incidents.json"),
    Path("../data/incidents.json"),
    Path("incidents.json"),
    Path("LLM-cybersecurity-analyzer/data/incidents.json"),
    Path("/content/LLM-cybersecurity-analyzer/data/incidents.json"),
]
DATA_PATH = next((path for path in DATA_PATH_CANDIDATES if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Shared incidents dataset not found. Clone the repository or upload data/incidents.json to the runtime."
    )

def load_incidents(data_path: Path) -> list[dict]:
    return json.loads(data_path.read_text(encoding="utf-8"))


incidents = load_incidents(DATA_PATH)

df_incidents = pd.DataFrame(
    {
        "id": [incident["id"] for incident in incidents],
        "incident_type": [incident["expected_extraction"]["incident_type"] for incident in incidents],
        "affected_asset": [incident["expected_extraction"]["affected_asset"] for incident in incidents],
    }
)

print(f"Total de incidentes no dataset: {len(df_incidents)}")
print(f"Fonte do dataset: {DATA_PATH}")
df_incidents


---
## 3. Tarefa 1 — Sumarização de Incidentes

Relatos de incidentes em ambientes reais frequentemente chegam em formato longo e narrativo. A sumarização automática ajuda o analista a entender o núcleo do caso sem ler o texto inteiro antes da triagem.

Utilizamos o modelo **`facebook/bart-large-cnn`**, um modelo encoder-decoder pré-treinado para tarefas de sumarização em inglês.


In [5]:
summarization_model_name = "facebook/bart-large-cnn"
summarization_tokenizer = AutoTokenizer.from_pretrained(summarization_model_name)
summarization_model = AutoModelForSeq2SeqLM.from_pretrained(summarization_model_name)
summarization_model.eval()

if torch.cuda.is_available():
    summarization_model = summarization_model.to("cuda")


def summarize_text(text, max_length=60, min_length=15):
    inputs = summarization_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024,
    )
    model_device = next(summarization_model.parameters()).device
    inputs = {k: v.to(model_device) for k, v in inputs.items()}

    with torch.no_grad():
        summary_ids = summarization_model.generate(
            **inputs,
            max_length=max_length,
            min_length=min_length,
            num_beams=4,
            early_stopping=True,
        )

    return summarization_tokenizer.decode(summary_ids[0], skip_special_tokens=True)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [6]:
example_incident = incidents[0]
summary = summarize_text(example_incident["text"])

print("INCIDENTE ORIGINAL:")
print_wrapped(example_incident["text"])
print()
print("SUMARIZACAO GERADA:")
print_wrapped(summary)


INCIDENTE ORIGINAL:
On June 14th, multiple employees in the Finance department reported receiving an email claiming to
be from the CFO, requesting urgent wire transfers to a new vendor account. The email contained a
spoofed domain (cfo-finance-corp.com) and urged recipients to act within one hour to avoid 'contract
penalties'. One employee clicked the link and entered their corporate credentials on a fake Office
365 login page before realizing the site was illegitimate. IT Security was notified and the
employee's account was immediately disabled pending investigation.

SUMARIZACAO GERADA:
On June 14th, multiple employees in the Finance department reported receiving an email claiming to
be from the CFO. The email contained a spoofed domain (cfo-finance-corp.com) and urged recipients to
act within one hour to avoid 'contract penalties' One


In [ ]:
summaries = []
for incident in incidents:
    summaries.append(
        summarize_text(
            incident["text"],
            max_length=50,
            min_length=10,
        )
    )

summary_df = pd.DataFrame(
    {
        "id": [incident["id"] for incident in incidents],
        "incident_type": [incident["expected_extraction"]["incident_type"] for incident in incidents],
        "summary": summaries,
    }
)

summary_df


,id,incident_type,summary
0,1,Phishing,"On June 14th, multiple employees in the Financ..."
1,2,Ransomware,Thousands of files were encrypted and renamed ...
2,3,Malware,A marketing department laptop was targeted by ...
3,4,Credential Theft,Security monitoring detected that an employee'...
4,5,Insider Threat,A system administrator exported a large volume...
5,6,Brute Force Attack,The authentication logs for the company's cust...
6,7,DDoS,The company's primary e-commerce website becam...
7,8,Phishing,A help desk technician received a phone call f...
8,9,Malware,Anti-virus software quarantined a file disguis...
9,10,Credential Theft,Several customer support agents reported that ...


---
## 4. Tarefa 2 — Extração com NER

Nesta etapa, usamos um modelo **NER pré-treinado** para identificar entidades diretamente no texto do incidente.

Aqui a saída é mantida de forma simples: mostramos **o retorno do próprio pipeline de NER**, sem regras adicionais para montar campos estruturados.


In [9]:
ner_model_name = "dslim/bert-base-NER"
ner_extractor = pipeline(
    "token-classification",
    model=ner_model_name,
    aggregation_strategy="simple",
    device=device,
)


def extract_named_entities(text):
    return ner_extractor(text)


config.json:   0%|          | 0.00/829 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


tokenizer_config.json:   0%|          | 0.00/59.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


In [10]:
for incident_id in [1, 4, 9]:
    incident = next(item for item in incidents if item["id"] == incident_id)
    extracted_entities = extract_named_entities(incident["text"])

    print(f"INCIDENTE #{incident_id}")
    print_wrapped(incident["text"])
    print()
    print("SAIDA DO NER:")
    for entity in extracted_entities:
        print(entity)
    print("-" * 80)


INCIDENTE #1
On June 14th, multiple employees in the Finance department reported receiving an email claiming to
be from the CFO, requesting urgent wire transfers to a new vendor account. The email contained a
spoofed domain (cfo-finance-corp.com) and urged recipients to act within one hour to avoid 'contract
penalties'. One employee clicked the link and entered their corporate credentials on a fake Office
365 login page before realizing the site was illegitimate. IT Security was notified and the
employee's account was immediately disabled pending investigation.

SAIDA DO NER:
{'entity_group': 'ORG', 'score': np.float32(0.86613643), 'word': 'Finance department', 'start': 40, 'end': 58}
{'entity_group': 'ORG', 'score': np.float32(0.7444848), 'word': 'CF', 'start': 111, 'end': 113}
{'entity_group': 'MISC', 'score': np.float32(0.9689775), 'word': 'Office 365', 'start': 389, 'end': 399}
{'entity_group': 'ORG', 'score': np.float32(0.99692655), 'word': 'IT Security', 'start': 455, 'end': 466}

In [11]:
ner_results = []
for incident in incidents:
    ner_results.append(
        {
            "id": incident["id"],
            "ner_entities": extract_named_entities(incident["text"]),
        }
    )

ner_results_df = pd.DataFrame(ner_results)
ner_results_df


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,id,ner_entities
0,1,"[{'entity_group': 'ORG', 'score': 0.86613643, ..."
1,2,"[{'entity_group': 'ORG', 'score': 0.51365846, ..."
2,3,[]
3,4,"[{'entity_group': 'ORG', 'score': 0.45450038, ..."
4,5,"[{'entity_group': 'MISC', 'score': 0.94700605,..."
5,6,[]
6,7,[]
7,8,[]
8,9,[]
9,10,"[{'entity_group': 'MISC', 'score': 0.97701454,..."


---
## 5. Leitura da Saída do NER

Cada item retornado pelo pipeline contém, em geral:

- `entity_group`: o tipo agregado da entidade reconhecida;
- `score`: a confiança do modelo;
- `word`: o trecho textual identificado;
- `start` e `end`: a posição da entidade no texto original.

Esse formato já é suficiente para inspecionar rapidamente quais pessoas, organizações, localidades e outros trechos relevantes o modelo reconheceu em cada relato.


In [12]:
entity_group_counts = {}

for row in ner_results:
    for entity in row["ner_entities"]:
        entity_group = entity["entity_group"]
        entity_group_counts[entity_group] = entity_group_counts.get(entity_group, 0) + 1

entity_group_df = pd.DataFrame(
    {
        "entity_group": list(entity_group_counts.keys()),
        "count": list(entity_group_counts.values()),
    }
).sort_values("count", ascending=False)

entity_group_df


,entity_group,count
1,MISC,8
0,ORG,6
2,LOC,2


---
## 6. Tokenização e Arquiteturas Relevantes

### Encoder-decoder

Modelos encoder-decoder são apropriados para tarefas de **transformação de sequência**, em que uma entrada textual precisa ser convertida em outra saída textual mais compacta ou estruturada. Neste notebook, usamos essa família para a **sumarização** com `facebook/bart-large-cnn`.

### Token classification / encoder-only

Modelos de token classification rotulam trechos do texto diretamente na entrada. Isso é útil para **Named Entity Recognition (NER)** e extração direta de entidades. Neste notebook, usamos essa família com `dslim/bert-base-NER`.

### Resumo comparativo

| Componente | Tipo de modelo | Papel na Fase 1 |
|---|---|---|
| `facebook/bart-large-cnn` | Encoder-decoder | Sumarizar relatos de incidentes |
| `dslim/bert-base-NER` | Token classification | Retornar entidades reconhecidas no texto |


---
## 7. Limitações Observadas Nesta Fase

Mesmo com o escopo mais enxuto desta fase, ainda existem limitações importantes.

### Cobertura de entidades

Um modelo NER genérico não foi treinado especificamente para o domínio de incidentes de segurança. Por isso, vários observáveis relevantes, como extensões maliciosas, nomes de técnicas ou artefatos específicos, podem não ser reconhecidos como entidades.

### Saída bruta do modelo

A saída do pipeline mostra somente as entidades detectadas pelo modelo, com seus rótulos e scores. Ela não organiza automaticamente essas entidades em um schema operacional de incidente.

### Conhecimento pré-treinado estático

Todos os modelos usados aqui dependem apenas do conhecimento aprendido no pré-treinamento. Eles não conhecem:

- playbooks internos da organização;
- inventário real de ativos críticos;
- incidentes passados da empresa;
- ameaças e indicadores publicados após o corte de treinamento.
